# Metadata Processing, Synapse and CellRanger

## 1. Metadata construction

### A. General metadata construction

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import os
import anndata as ad
import synapseclient
import synapseutils


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
FILE_INDIVIDUAL   = "/tscc/nfs/home/aopatel/synapse_meta_NEW/SEA-AD_individual_metadata.csv"
FILE_BIOSPECIMEN  = "/tscc/nfs/home/aopatel/synapse_meta_NEW/SEA-AD_biospecimen_metadata.csv"
FILE_SNRNASEQ     = "/tscc/nfs/home/aopatel/synapse_meta_NEW/SEA-AD_assay_snRNAseq_metadata.csv"
FILE_MANIFEST     = "/tscc/nfs/home/aopatel/synapse_meta_NEW/SEA-AD_file_manifest_metadata.csv"
OUT_CSV           = "/tscc/nfs/home/aopatel/synapse_meta_NEW/sea_ad_merged_metadata.csv"


In [ ]:
# ── 1. Load ────────────────────────────────────────────────────────────────────
print("Loading files...")
ind = pd.read_csv(FILE_INDIVIDUAL)
bio = pd.read_csv(FILE_BIOSPECIMEN)
snr = pd.read_csv(FILE_SNRNASEQ)
man = pd.read_csv(FILE_MANIFEST)

print(f"  Individual  : {ind.shape[0]} rows x {ind.shape[1]} cols")
print(f"  Biospecimen : {bio.shape[0]} rows x {bio.shape[1]} cols")
print(f"  snRNAseq    : {snr.shape[0]} rows x {snr.shape[1]} cols")
print(f"  Manifest    : {man.shape[0]} rows x {man.shape[1]} cols")

In [ ]:
# ── 2. Clean up shared junk columns ───────────────────────────────────────────
# All three files have a redundant "Unnamed: 0" index column — drop it.
# "assay" appears in snr, bio, man; rename before merging to preserve both.
for df in [ind, bio, snr, man]:
    df.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

snr.rename(columns={"assay": "assay_snr"}, inplace=True)
bio.rename(columns={"assay": "assay_bio"}, inplace=True)
man.rename(columns={"assay": "assay_man"}, inplace=True)


In [ ]:
# ── 2A. Filter biospecimen to snRNAseq rows only ───────────────────────────────
# The biospecimen file covers 5 assay types; we only need the 846 rows that
# have a matching snRNAseq record.  Filtering here avoids any fan-out risk
# from the 3 ATACSeq duplicate specimenIDs.
bio_snr = bio[bio["specimenID"].isin(snr["specimenID"])].copy()
print(f"\n  Biospecimen rows matching snRNAseq specimenIDs: {len(bio_snr)}")

In [ ]:
# ── 2B. Merge ───────────────────────────────────────────────────────────────────
#  Step A: snRNAseq ← Biospecimen  (on specimenID, 1:1)
merged = snr.merge(bio_snr, on="specimenID", how="left", validate="1:1")
assert len(merged) == len(snr), "Fan-out in snRNAseq → Biospecimen join!"

#  Step B: + Individual  (on individualID, many:1)
merged = merged.merge(ind, on="individualID", how="left", validate="m:1")
assert len(merged) == len(snr), "Row count changed in Biospecimen → Individual join!"

print(f"  Merged table: {merged.shape[0]} rows x {merged.shape[1]} cols  ✓")

In [ ]:
# ── 2C.  Filter for scRNAseq 

# keep only scrnaseq and actual .fastq.gz files
man_filtered = man[
    (man['assay_man'].str.lower() == 'scrnaseq') & 
    (man['file_name'].str.lower().str.endswith('.fastq.gz', na=False))
].copy()

# Remove duplicates based on 'file_name'
# keep='first' ensures we only keep the first occurrence of each filename # this is fine 
# each file_name has only one unique specimen ID (double check)
man_deduplicated = man_filtered.drop_duplicates(subset=['file_name'], keep='first')

# Get a count of unique files for each tissue
# This creates a Series where the index is 'tissue' and the value is the count
tissue_counts = man_deduplicated.groupby('tissue').size().reset_index(name='file_count')

# Display counts for a quick check
print(tissue_counts)



In [ ]:
# ── 2D.  Final merge between merged and man_deduplicated

# man_deduplicated → your filtered FASTQ DataFrame
# merged        → your specimen metadata DataFrame
 
# Column names for the join key
man_spec   = "specimen_id"
merge_spec = "specimenID"
 
# ── Collapse file names per specimen ──────────────────────────────────
files_collapsed = (
    man_deduplicated
    .groupby(man_spec)["file_name"]
    .agg(file_names=lambda x: "|".join(sorted(x)), file_count="count")
    .reset_index()
    .rename(columns={man_spec: merge_spec})  # align key name for merge
)
 
# ── Merge with specimen metadata ──────────────────────────────────────
merged_out = merged.merge(files_collapsed, on=merge_spec, how="left")
 
merged_out.head()

In [ ]:
# ── 2E. Add CPS and Severely Affected Donor from Gabitto et al. file ────────────────


# Load Gabitto et al. file — only need obs, not the full matrix
ref = ad.read_h5ad("/tscc/lustre/ddn/scratch/aopatel/mtg_reference/SEAAD_MTG_RNAseq_final-nuclei.2024-02-13.h5ad",
                  backed='r')

# Extract donor-level CPS and Severely Affected Donor
# One row per donor (deduplicated)
donor_meta = (
    ref.obs[['Donor ID', 'Continuous Pseudo-progression Score', 'Severely Affected Donor']]
    .drop_duplicates(subset='Donor ID')
    .rename(columns={
        'Continuous Pseudo-progression Score': 'CPS',
        'Donor ID': 'individualID'
    })
    .reset_index(drop=True)
)

print(f'Unique donors in reference: {len(donor_meta)}')
print(f'CPS null: {donor_meta["CPS"].isna().sum()}')
print(f'Severely Affected Donor null: {donor_meta["Severely Affected Donor"].isna().sum()}')
print(donor_meta.head())

# Merge into merged_out on individualID
before_cols = merged_out.shape[1]
merged_out = merged_out.merge(donor_meta, on='individualID', how='left')
after_cols = merged_out.shape[1]

print(f'\nColumns added: {after_cols - before_cols}')
print(f'Donors with CPS: {merged_out["CPS"].notna().sum()}')
print(f'Donors WITHOUT CPS: {merged_out["CPS"].isna().sum()}')

In [ ]:
print(f'merged rows: {len(merged)}')
print(f'merged_out rows: {len(merged_out)}')  # should still be 846
print(f'merged_out cols: {merged_out.shape[1]}')

# Check how many specimens got file_names matched
print(f'Specimens with file_names: {merged_out["file_names"].notna().sum()}')
print(f'Specimens WITHOUT file_names: {merged_out["file_names"].isna().sum()}')

# Check file_count distribution
print(f'\nFile count distribution:')
print(merged_out["file_count"].value_counts())

In [ ]:
# ── 3. Filter and derived columns ─────────────────────────────────────────────

# Keep only individuals with a Consensus clinical diagnosis of
#     'Control' or 'Alzheimers disease' (exact match).
keep_dx = ["Control", "Alzheimers disease"]
before      = len(merged_out)
merged_out  = merged_out[merged_out["Consensus clinical diagnosis"].isin(keep_dx)].copy()
after       = len(merged_out)
print(f"\n  Filtered to Control + Alzheimers disease: {before} → {after} rows "
      f"({before - after} dropped)")
print(f"  Donors retained: {merged_out['individualID'].nunique()}")
print(f"  Breakdown:\n{merged_out['Consensus clinical diagnosis'].value_counts().to_string()}")

# ageDeath as numeric
#     "90+" → 91  (flagged separately via ageDeath_censored)
merged_out["ageDeath_numeric"] = (
    merged_out["ageDeath"]
    .str.replace("90+", "91", regex=False)
    .astype(float)
)
merged_out["ageDeath_censored"] = merged_out["ageDeath"].str.contains(r"\+", regex=True)

# Remove any sample with RIN <5
merged_out = merged_out[(merged_out["RIN"] >= 5) | merged_out["RIN"].isna()].copy()

# Comparison group label
merged_out["comparison_group"] = merged_out["Consensus clinical diagnosis"].map({
    "Control"            : "PathologyControl",
    "Alzheimers disease" : "AD",
})

In [ ]:
print(f'\nAfter RIN filter: {len(merged_out)} rows')
print(f'RIN < 5 dropped: {after - len(merged_out)}')
print(f'\ncomparison_group breakdown:')
print(merged_out['comparison_group'].value_counts())
print(f'\nDonors: {merged_out["individualID"].nunique()}')
print(f'Specimens: {len(merged_out)}')

In [ ]:
# ── 4. Column ordering ─────────────────────────────────────────────────────────
priority_cols = [
    # Identity
    "individualID", "specimenID",
    # Group labels
    "comparison_group", "ADNC", "Consensus clinical diagnosis", "diagnosis",
    # Neuropath scores
    "CERAD", "Braak", "Thal phase", "Overall CAA Score",
    "Lewy body disease pathology", "LATE-NC stage",
    # Demographics
    "sex", "race", "ageDeath", "ageDeath_numeric", "ageDeath_censored",
    "yearsEducation", "pmi", "pH", "brainWeight",
    # Genetics
    "apoeGenotype", "apoe4Status",
    # Tissue
    "tissue", "BrodmannArea", "organ",
    # Cognitive
    "Cognitive status", "CASI score", "MMSE score", "MoCA score",
    # snRNAseq QC
    "numberCells", "medianGenes", "medianUMIs",
    "totalReads", "validBarcodeReads", "RIN",
    "platform", "libraryPrep", "libraryPreparationMethod",
    "rnaBatch", "libraryBatch", "sequencingBatch",
    "readLength", "runType",
    # FASTQ linkage key
    "specimen_suffix",
    # Assay labels
    "assay_snr", "assay_bio",
]
remaining  = [c for c in merged_out.columns if c not in priority_cols]
merged_out = merged_out[priority_cols + remaining]

In [ ]:
# Check for any priority columns missing from merged_out
missing_cols = [c for c in priority_cols if c not in merged_out.columns]
if missing_cols:
    print(f'WARNING - these priority cols not in merged_out: {missing_cols}')
else:
    print('All priority columns present ✅')

print(f'Final shape: {merged_out.shape}')
print(f'First 5 columns: {list(merged_out.columns[:5])}')

In [ ]:
# ── 5. Validation report ───────────────────────────────────────────────────────
print("\n── Validation ────────────────────────────────────────")
print(f"  Final shape          : {merged_out.shape}")
print(f"  Null individualIDs   : {merged_out['individualID'].isna().sum()}")
print(f"  Null specimenIDs     : {merged_out['specimenID'].isna().sum()}")
print(f"  Null tissue          : {merged_out['tissue'].isna().sum()}")
print(f"  Null ADNC            : {merged_out['ADNC'].isna().sum()}")

print("\n── Group counts ──────────────────────────────────────")
grp = merged_out.groupby("comparison_group", observed=True)
spec_counts   = grp["specimenID"].count()
donor_counts  = grp["individualID"].nunique()
tissue_counts = grp["tissue"].nunique()
summary = pd.DataFrame({
    "specimens": spec_counts,
    "donors"   : donor_counts,
    "tissues"  : tissue_counts,
})
print(summary.to_string())

print("\n── Specimens per tissue per group ────────────────────")
pivot = merged_out.pivot_table(
    index="tissue",
    columns="comparison_group",
    values="specimenID",
    aggfunc="count",
    fill_value=0,
    observed=True,
)
print(pivot.to_string())

print("\n── Donors per tissue per group ───────────────────────")
pivot_donors = merged_out.pivot_table(
    index="tissue",
    columns="comparison_group",
    values="individualID",
    aggfunc="nunique",
    fill_value=0,
    observed=True,
)
print(pivot_donors.to_string())

print("\n── ADNC × Consensus clinical dx (full cohort) ────────")
cross = pd.crosstab(
    merged_out["ADNC"].fillna("NA"),
    merged_out["Consensus clinical diagnosis"].fillna("Unknown"),
)
print(cross.to_string())

print("\n── Grey-zone donors (clinical Control but ADNC Intermediate/High) ──")
grey = merged_out[
    (merged_out["Consensus clinical diagnosis"] == "Control") &
    (merged_out["ADNC"].isin(["Intermediate", "High"]))
][["individualID", "ADNC", "Braak", "CERAD", "Thal phase", "Cognitive status"]].drop_duplicates()
print(f"  {grey['individualID'].nunique()} donors, {len(grey)} unique donor rows")
print(grey.to_string(index=False))

print("\n── Sex breakdown by tissue and diagnosis ─────────────")
sex_pivot = merged_out.pivot_table(
    index=["tissue", "comparison_group"],
    columns="sex",
    values="individualID",
    aggfunc="nunique",
    fill_value=0,
    observed=True,
)
print(sex_pivot.to_string())

print("\n── Sex breakdown by diagnosis only ───────────────────")
sex_dx = merged_out.groupby(["comparison_group", "sex"], observed=True)["individualID"].nunique().unstack(fill_value=0)
print(sex_dx.to_string())

In [ ]:
# ── 6. QC metric breakdown: RIN, pmi, pH ─────────────────────────────────────────
qc_cols = ["RIN", "pmi", "pH"]

# Ensure numeric
for col in qc_cols:
    merged_out[col] = pd.to_numeric(merged_out[col], errors="coerce")

# Overall summary
print("\n── Overall descriptive stats ─────────────────────────────────────────")
print(merged_out[qc_cols].describe().round(2).to_string())

# Null counts
print("\n── Missing values ────────────────────────────────────────────────────")
for col in qc_cols:
    print(f"  {col}: {merged_out[col].isna().sum()} null ({merged_out[col].isna().mean()*100:.1f}%)")

# Breakdown by comparison group
print("\n── By comparison_group ───────────────────────────────────────────────")
print(
    merged_out.groupby("comparison_group", observed=True)[qc_cols]
    .agg(["mean", "median", "std", "min", "max"])
    .round(2)
    .to_string()
)

# Breakdown by tissue
print("\n── By tissue ─────────────────────────────────────────────────────────")
print(
    merged_out.groupby("tissue", observed=True)[qc_cols]
    .agg(["mean", "median", "std"])
    .round(2)
    .to_string()
)

# Breakdown by tissue and diagnosis — mean ± std for RIN, PMI, pH
print("\n── RIN, PMI, pH by tissue and diagnosis ──────────────────────────────")
for col in qc_cols:
    print(f"\n  {col}:")
    summary = (
        merged_out.groupby(["tissue", "comparison_group"], observed=True)[col]
        .agg(mean="mean", std="std", n="count")
        .round(2)
    )
    summary["mean ± std"] = summary.apply(
        lambda r: f"{r['mean']} ± {r['std']}", axis=1
    )
    print(summary[["mean ± std", "n"]].to_string())

In [ ]:
# ── 6A. Age breakdown by diagnosis for target regions ─────────────────────────────
print("\n── Mean age ± std by diagnosis (MTG, STG, ITG, DLPFC only) ──────────")

target_tissues = [
    "middle temporal gyrus",
    "superior temporal gyrus",
    "inferior temporal gyrus",
    "dorsolateral prefrontal cortex"
]

age_summary = (
    merged_out[merged_out["tissue"].isin(target_tissues)]
    .drop_duplicates(subset=["individualID", "tissue"])  # one row per donor per tissue
    .groupby(["tissue", "comparison_group"], observed=True)["ageDeath_numeric"]
    .agg(
        mean="mean",
        std="std",
        n="count"
    )
    .round(2)
)

age_summary["mean ± std"] = age_summary.apply(
    lambda r: f"{r['mean']} ± {r['std']}", axis=1
)

print(age_summary[["mean ± std", "n"]].to_string())

In [ ]:
# ── 7. Save ────────────────────────────────────────────────────────────────────

merged_out.to_csv(OUT_CSV, index=False)
print(f"\n✓ Saved CSV  → {OUT_CSV}")

In [ ]:
# Verify the saved file
verify = pd.read_csv(OUT_CSV)
print(f'Rows: {verify.shape[0]}')
print(f'Cols: {verify.shape[1]}')
print(f'Columns: {list(verify.columns[:5])}')
assert verify.shape == merged_out.shape, "Shape mismatch after save!"
print('✅ File saved and verified correctly')

### B. Tissue specific metadata construction

In [ ]:
# ── 8. Make tissue specific ───────────────────────────────────────────────────

# Find tissues
print(merged_out["tissue"].unique())  # check exact tissue name first

In [ ]:
# ── 8A. Subset ───────────────────────────────────────────────────

merged_out = merged_out[merged_out["tissue"] == "dorsolateral prefrontal cortex"].copy()
print(f"  DLPFC subset: {len(merged_out)} specimens, {merged_out['individualID'].nunique()} donors")

merged_out.to_csv('/tscc/nfs/home/aopatel/synapse_meta_NEW/DLPFC_sea_ad_merged_metadata.csv', index=False)

In [ ]:
# ── 8B. CPS availability after brain region filter ───────────────
print(f'\nAfter brain region filter:')
print(f'  Specimens with CPS: {merged_out["CPS"].notna().sum()}')
print(f'  Specimens WITHOUT CPS: {merged_out["CPS"].isna().sum()}')

print(f'\nCPS distribution (overall):')
print(merged_out["CPS"].describe().round(3))

print(f'\nCPS distribution by comparison_group:')
print(
    merged_out.groupby("comparison_group", observed=True)["CPS"]
    .describe()
    .round(3)
    .to_string()
)

print(f'\nCPS null by comparison_group:')
print(merged_out.groupby("comparison_group", observed=True)["CPS"].apply(lambda x: x.isna().sum()))

In [ ]:
# ── 9. Find Synapse IDs for each fastq of interest ─────────────────────────────
syn = synapseclient.login()

# Load metadata
metadata = pd.read_csv('/tscc/nfs/home/aopatel/synapse_meta_NEW/DLPFC_sea_ad_merged_metadata.csv')

# Target R1, R2, and I1 (exclude I2)
target_files = (
    metadata['file_names']
    .str.split('|')
    .explode()
    .str.strip()
    .pipe(lambda s: s[s.str.contains('_R1_|_R2_|_I1_')])
    .unique()
    .tolist()
)
print(f'Target files (R1+R2+I1): {len(target_files)}')  # should be ~300

# Search both folders
folder_ids = {
    'DLPFC_upload_1': 'syn26273710',
    'DLPFC_upload_2': 'syn51792375',
}

all_files = []
for folder_name, folder_id in folder_ids.items():
    children = syn.getChildren(folder_id, includeTypes=['file'])
    folder_files = [{'id': c['id'], 'name': c['name'], 'folder': folder_name}
                    for c in children]
    all_files.extend(folder_files)

folder_df = pd.DataFrame(all_files)

# Match
matched = folder_df[folder_df['name'].isin(target_files)]
missing = set(target_files) - set(matched['name'])
print(f'Matched: {len(matched)}')
print(f'Missing: {len(missing)}')

# ── Create syn_id_list.txt ─────────────────────────────────────────────────────
matched['id'].to_csv('/tscc/nfs/home/aopatel/synapse_meta_NEW/syn_id_list.txt', index=False, header=False)

# Verify using same path
with open('/tscc/nfs/home/aopatel/synapse_meta_NEW/syn_id_list.txt') as f:
    lines = f.readlines()
print(f'Lines in syn_id_list.txt: {len(lines)}')

In [ ]:
print("✅ Use this number for #SBATCH --array=1- :", len(lines),"🚨")

In [ ]:
# ── 10. Check synapse folders for other brain regions ─────────────────────────────
# Get parent of upload 1 to find all sibling upload folders
upload1 = syn.get('syn26273710', downloadFile=False)
print('Parent of upload 1:', upload1.parentId)

# List all folders under that parent
children = list(syn.getChildren(upload1.parentId, includeTypes=['folder']))
for c in children:
    print(c['id'], c['name'])

## 2. Fastq download and verification instructions

In [ ]:
# ── 11. Job array script to download fastq files  ─────────────────────────────

# 🚨 Make sure to change paths, job array number, brain region in files (this one and download_fastq.sh)

# In terminal of TSCC run:
# mkdir -p logs
# sbatch download_fastq.sh



In [ ]:
# ── 11A. After all fastq files have been downloaded   ─────────────────────────────

metadata = pd.read_csv('/tscc/nfs/home/aopatel/synapse_meta_NEW/DLPFC_sea_ad_merged_metadata.csv')
expected = set(
    metadata['file_names']
    .str.split('|')
    .explode()
    .str.strip()
    .pipe(lambda s: s[s.str.contains('_R1_|_R2_|_I1_')])
    .unique()
    .tolist()
)

downloaded = set(os.listdir('/tscc/lustre/ddn/scratch/aopatel/DLPFC_fastq'))

print(f'Expected: {len(expected)}')
print(f'Downloaded: {len(downloaded)}')
print(f'Missing: {len(expected - downloaded)}')
for f in sorted(expected - downloaded):
    print(f'  {f}')
print(f'Unexpected: {len(downloaded - expected)}')

In [ ]:
# ── 11B. Check md5 sums  ─────────────────────────────
# 🚨 Run (it will print results into same log directory but not overite it as you have not generated md5 logs yet):
# 🚨 Remember to ensure array number is specific for the number files in your brain region
# sbatch md5check.sh

# Run these commands to check pass/fail
# grep "FAIL" logs/md5_*.out
# grep "PASS" logs/md5_*.out | wc -l


## 3. CellRanger instructions

In [ ]:
# ── 12A. Installation guidelines  ─────────────────────────────

In [ ]:
#### ___________________________ CellRanger ___________________________

# 🚨 Install Cellranger 10.0.0

# curl -o cellranger-10.0.0.tar.gz "https://cf.10xgenomics.com/releases/cell-exp/cellranger-10.0.0.tar.gz?Expires=1774497438&Key-Pair-Id=APKAI7S6A5RYOXBWRPDA&Signature=RneA3LV2FE8GWzSpnNTPWNXLU0XuAsQMtKT4CilmViIIWsOkzm5GjAymBVPzRsHvLiZ-ZOXyQHBaDrCVLrGTed2lPXJxImUPsscz9F6I616MWZSeRrOagdNRK0i5Gz6THjwEzxBq3ZepDv~7qvuSOGiIjcYdJk0qFBU6LZvaRW-s-3qSEqCrv2E25U7a0MnL5oC1BCVAMRaG1V4gc8qbvA3VVDl9j0y67tBbqdEUcB7UeGB73KpPWxgdoveCDfFZsQAekABSHBMLWbfM7MOiYiyi~MMrXI47hz1ujbfJm7k98n9rx-ABRtoRGvu0VBgM5pZFcOraznouJFz1YOvTzA__"
# tar -xzvf cellranger-10.0.0.tar.gz

# 🚨 Make it part of path!

# echo 'export PATH=~/cellranger-10.0.0:$PATH' >> ~/.bashrc
# source ~/.bashrc
# cellranger --version


# 🚨 md5sum and remove .tar.gz file

# md5sum ~/cellranger-10.0.0.tar.gz # verify with CellRanger downloads websites. It's on there!
# rm cellranger-10.0.0.tar.gz

#### _____________________ Human Genome Reference _____________________

# 🚨 Install Human Genome Reference GRCh38 2024-A gex

# curl -O "https://cf.10xgenomics.com/supp/cell-exp/refdata-gex-GRCh38-2024-A.tar.gz"
# tar -xzvf refdata-gex-GRCh38-2024-A.tar.gz

# 🚨 md5sum and remove .tar.gz file

# md5sum ~/refdata-gex-GRCh38-2024-A.tar.gz
# rm refdata-gex-GRCh38-2024-A.tar.gz



In [ ]:
# ── 12B. Create sample list for CellRanger   ─────────────────────────────

In [ ]:
# Extract unique sample prefixes from filenames
fastq_dir = '/tscc/lustre/ddn/scratch/aopatel/DLPFC_fastq'
files = os.listdir(fastq_dir)

samples = sorted(set(
    f.split('_S01_')[0] for f in files if f.endswith('.fastq.gz')
))

print(f'Total samples: {len(samples)}')
for s in samples[:5]:
    print(s)

# Save
with open('/tscc/nfs/home/aopatel/synapse_meta_NEW/sample_list.txt', 'w') as f: 
    f.write('\n'.join(samples))

In [ ]:
# 🚨 In bash, ensure lines in this file match the total number of samples,
# this is important for the job array

#wc -l /tscc/nfs/home/aopatel/synapse_meta_NEW/sample_list.txt


# often it wont and you just need to add a new line character '\n' that 
# should be the only issue. If you need to add a new character run 
# with open()...


with open('/tscc/nfs/home/aopatel/synapse_meta_NEW/sample_list.txt', 'w') as f:
    f.write('\n'.join(samples) + '\n')

In [ ]:
# ── 12C. Ensure metadata file names ALL match ─────────────────────────────

metadata = pd.read_csv('/tscc/nfs/home/aopatel/synapse_meta_NEW/DLPFC_sea_ad_merged_metadata.csv')

# Get specimen IDs from metadata
specimen_ids = set(metadata['specimenID'].unique())

# Get sample prefixes from file_names
file_samples = set(
    metadata['file_names']
    .str.split('|')
    .explode()
    .str.strip()
    .str.split('_S01_')
    .str[0]
    .unique()
)

# Load sample_list
with open('/tscc/nfs/home/aopatel/synapse_meta_NEW/sample_list.txt') as f:
    sample_list = set(f.read().splitlines())

print(f'Samples in sample_list.txt: {len(sample_list)}')
print(f'Samples in metadata file_names: {len(file_samples)}')
print(f'Match: {sample_list == file_samples}')
print(f'Missing from sample_list: {file_samples - sample_list}')
print(f'Extra in sample_list: {sample_list - file_samples}')

In [ ]:
# ── 12C. After CellRanger   ─────────────────────────────

In [ ]:
# 🚨 To check for successful completions, in bash run:

# grep -l "Pipestance completed successfully" logs/cr_*.out | wc -l

# This number must match the number of samples present for the brain region (eg. 100 for DLFPC)

# 🚨 If lock issues are found:

# 1. Ensure no other Cell Ranger instance is running for that sample
# 2. 🚨 Delete the lock file: rm ${OUT_DIR}/${sample}/_lock
#    (This is the officially recommended solution by 10x Genomics)
# 3. Create rerun_samples.txt with affected samples
# 4. Submit new job array — Cell Ranger will resume from checkpoints if available
